
# Exp40 — Validation-Calibrated PRR Trust (No Retraining / No Forecaster Inference)

This experiment tests whether the fixed downstream trust coefficient \(\beta=0.1\) is unnecessarily restrictive.

For each condition, PRR is integrated with the frozen direct forecast as

\[
\hat{\mathbf y}^{\,A}
=
\hat{\mathbf y}^{\,D}
+
\beta\left(
\hat{\mathbf y}^{\,R}-\hat{\mathbf y}^{\,D}
\right),
\]

where:

- \(\hat{\mathbf y}^{D}\): frozen direct forecast,
- \(\hat{\mathbf y}^{R}\): PRR retrieval forecast,
- \(\beta\): trust assigned to PRR.

## Protocol

We compare three settings:

1. **Fixed**: \(\beta=0.1\) — current paper control.
2. **Global-Cal**: one \(\beta\) selected from validation over all 120 conditions.
3. **Dataset-Cal**: one \(\beta\) per dataset selected from validation over its 20 conditions.

The calibration grid is

\[
\beta\in\{0, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20\}.
\]

Selection uses **validation only**. Test data are never used to choose \(\beta\).

To prevent large-scale datasets or horizons from dominating the shared-\(\beta\) selection, the validation objective is the macro-average normalized error

\[
J(\beta)=
\frac{1}{|\mathcal C|}
\sum_{c\in\mathcal C}
\frac{\mathrm{MSE}_{c}(\beta)}
{\mathrm{MSE}^{\mathrm{Direct}}_{c}}.
\]

Thus each condition contributes equally.

## Important

- No PRR retraining.
- No forecaster retraining.
- No forecaster inference.
- No test-set hyperparameter selection.
- Existing Exp33 manifests, direct caches, and PRR retrieval caches are reused exactly.


In [ ]:

# Cell 1 — setup

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PRR_ROOT", str(Path.cwd().resolve().parents[1] if Path.cwd().name == "prr_long_horizon" else Path.cwd().resolve())))

MANIFEST_ROOT = ROOT / "results" / "exp33" / "final_eval_manifest"
PATCHTST_DIRECT_ROOT = ROOT / "results" / "exp34" / "patchtst_direct_cache"
OTHER_DIRECT_ROOT = ROOT / "results" / "exp37D" / "direct_cache"

# PRR retrieval cache (formerly the PRR cache)
PRR_PART_ROOT = ROOT / "results" / "exp35B" / "channel_parts"

# Optional exact fixed-beta reference from Exp38R3
EXP38_FIXED = ROOT / "results" / "exp38R" / "fixed_beta01_120_conditions.csv"

OUT = ROOT / "results" / "exp40_calibrated_prr"
OUT.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)
BACKBONES = ("PatchTST","iTransformer","TimeMixer","SegMoE","DLinear")
SPLITS = ("val","test")

BETA_GRID = np.array([0.0, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20], dtype=np.float64)
FIXED_BETA = 0.10
EPS = 1e-12

print("ROOT:", ROOT)
print("OUT :", OUT)
print("Beta grid:", BETA_GRID.tolist())

assert MANIFEST_ROOT.exists(), MANIFEST_ROOT
assert PRR_PART_ROOT.exists(), PRR_PART_ROOT


In [ ]:

# Cell 2 — exact Exp33 manifest loader

def load_manifest(dataset, horizon, split):
    p = MANIFEST_ROOT / f"{dataset}_H{horizon}_{split}.npz"
    assert p.exists(), p

    with np.load(p, allow_pickle=False) as z:
        a = np.asarray(z["anchor"], np.int64)
        c = np.asarray(z["channel"], np.int64)

    assert a.shape == c.shape
    return a, c

rows = []
for d in DATASETS:
    for h in HORIZONS:
        for sp in SPLITS:
            a, c = load_manifest(d, h, sp)
            rows.append({
                "dataset": d,
                "horizon": h,
                "split": sp,
                "pairs": len(a),
                "channels": len(np.unique(c)),
            })

manifest_df = pd.DataFrame(rows)
display(manifest_df)

assert len(manifest_df) == 48
print("[PASS] Exp33 manifests = 48/48")


In [ ]:

# Cell 3 — exact direct-cache loader

def direct_slug(backbone):
    return backbone.lower().replace("-", "").replace("_", "")

def direct_path_candidates(backbone, dataset, horizon, split):
    root = PATCHTST_DIRECT_ROOT if backbone == "PatchTST" else OTHER_DIRECT_ROOT
    slug = direct_slug(backbone)

    exact = [
        root / f"{dataset}_H{horizon}_{split}_{slug}_direct.npz",
        root / f"{dataset}_H{horizon}_{split}_{backbone}_direct.npz",
        root / f"{dataset}_H{horizon}_{split}_direct.npz",
    ]
    out = [p for p in exact if p.exists()]

    if not out and root.exists():
        for p in root.rglob("*.npz"):
            s = p.name.lower()
            if dataset.lower() in s and f"h{horizon}" in s and split in s:
                out.append(p)

    return out

def load_direct(backbone, dataset, horizon, split):
    ma, mc = load_manifest(dataset, horizon, split)

    for p in direct_path_candidates(backbone, dataset, horizon, split):
        try:
            with np.load(p, allow_pickle=False) as z:
                if not {"anchor", "channel", "direct", "truth"}.issubset(z.files):
                    continue

                a = np.asarray(z["anchor"], np.int64)
                c = np.asarray(z["channel"], np.int64)
                d = np.asarray(z["direct"], np.float32)
                y = np.asarray(z["truth"], np.float32)

            ok = (
                np.array_equal(a, ma)
                and np.array_equal(c, mc)
                and d.shape == (len(ma), horizon)
                and y.shape == (len(ma), horizon)
                and np.isfinite(d).all()
                and np.isfinite(y).all()
            )
            if not ok:
                continue

            return {
                "anchor": a,
                "channel": c,
                "direct": d,
                "truth": y,
                "path": str(p),
            }

        except Exception:
            continue

    raise FileNotFoundError(
        f"Exact direct cache not found: {backbone} {dataset} H={horizon} {split}"
    )

audit = []
for b in BACKBONES:
    for d in DATASETS:
        for h in HORIZONS:
            ok = True
            paths = []
            for sp in SPLITS:
                try:
                    x = load_direct(b, d, h, sp)
                    paths.append(x["path"])
                except Exception:
                    ok = False
            audit.append({
                "backbone": b,
                "dataset": d,
                "horizon": h,
                "ok": ok,
            })

direct_audit = pd.DataFrame(audit)
display(
    direct_audit.groupby("backbone", as_index=False)
    .agg(total=("ok", "size"), ok=("ok", "sum"))
)

assert int(direct_audit.ok.sum()) == 120
print("[PASS] exact direct val/test cache = 120/120 conditions")


In [ ]:

# Cell 4 — exact PRR per-channel retrieval assembler

def prr_channel_file(dataset, horizon, split, ch):
    return (
        PRR_PART_ROOT
        / dataset
        / f"H{horizon}"
        / split
        / f"ch{int(ch):04d}.npz"
    )

def load_prr(dataset, horizon, split):
    ma, mc = load_manifest(dataset, horizon, split)

    out = np.full((len(ma), horizon), np.nan, dtype=np.float32)
    used = []

    for ch in np.unique(mc):
        rows = np.flatnonzero(mc == ch)
        expected_anchor = ma[rows]

        p = prr_channel_file(dataset, horizon, split, int(ch))
        assert p.exists(), p

        with np.load(p, allow_pickle=False) as z:
            assert "anchor" in z.files and "retrieval" in z.files
            a = np.asarray(z["anchor"], np.int64)
            r = np.asarray(z["retrieval"], np.float32)

        assert r.shape == (len(a), horizon), (p, r.shape)

        if np.array_equal(a, expected_anchor):
            order = np.arange(len(a), dtype=np.int64)
        else:
            # Robust anchor matching fallback.
            pos = {}
            for j, v in enumerate(a):
                pos.setdefault(int(v), []).append(j)

            order = []
            for v in expected_anchor:
                bucket = pos.get(int(v), [])
                if not bucket:
                    raise RuntimeError(
                        f"PRR cache mismatch: {dataset} H={horizon} "
                        f"{split} ch={ch}, missing anchor={int(v)}"
                    )
                order.append(bucket.pop(0))
            order = np.asarray(order, np.int64)

        out[rows] = r[order]
        used.append(str(p))

    assert out.shape == (len(ma), horizon)
    assert np.isfinite(out).all()

    return {
        "anchor": ma,
        "channel": mc,
        "retrieval": out,
        "files": used,
    }

# Fast preflight: verify all 48 split/condition assemblies exist.
rows = []
for d in DATASETS:
    for h in HORIZONS:
        for sp in SPLITS:
            x = load_prr(d, h, sp)
            rows.append({
                "dataset": d,
                "horizon": h,
                "split": sp,
                "pairs": len(x["anchor"]),
                "source_files": len(x["files"]),
            })

prr_audit = pd.DataFrame(rows)
display(prr_audit.groupby("split", as_index=False).agg(
    conditions=("dataset", "size"),
    source_files=("source_files", "sum"),
))

assert len(prr_audit) == 48
print("[PASS] exact PRR val/test retrieval = 48/48")


In [ ]:

# Cell 5 — metrics / integration helpers

def mse(a, b):
    a = np.asarray(a, np.float64)
    b = np.asarray(b, np.float64)
    return float(np.mean((a - b) ** 2))

def mae(a, b):
    a = np.asarray(a, np.float64)
    b = np.asarray(b, np.float64)
    return float(np.mean(np.abs(a - b)))

def combine(direct, retrieval, beta):
    d = np.asarray(direct, np.float32)
    r = np.asarray(retrieval, np.float32)
    return d + float(beta) * (r - d)

def gain_pct(direct_mse, method_mse):
    return 100.0 * (direct_mse - method_mse) / direct_mse

def wtl(series, tol=1e-10):
    x = pd.Series(series).dropna().to_numpy(float)
    return {
        "W": int((x > tol).sum()),
        "T": int((np.abs(x) <= tol).sum()),
        "L": int((x < -tol).sum()),
    }

print("[PASS] helpers ready")



## Validation calibration objective

For each validation condition \(c\),

\[
\rho_c(\beta)
=
\frac{\mathrm{MSE}_{c}(\beta)}
{\mathrm{MSE}^{D}_{c}}.
\]

The shared calibration objective is the macro-average

\[
J_G(\beta)=\frac1{120}\sum_{c=1}^{120}\rho_c(\beta),
\]

and for dataset-specific calibration,

\[
J_d(\beta)=\frac1{20}\sum_{c\in d}\rho_c(\beta).
\]

This gives every backbone–dataset–horizon condition equal weight.


In [ ]:

# Cell 6 — build validation beta curves (NO test use)

val_rows = []

for d in DATASETS:
    for h in HORIZONS:
        prr = load_prr(d, h, "val")["retrieval"]

        for b in BACKBONES:
            dv = load_direct(b, d, h, "val")
            direct = dv["direct"]
            truth = dv["truth"]

            dm = mse(direct, truth)

            for beta in BETA_GRID:
                pred = combine(direct, prr, beta)
                mm = mse(pred, truth)

                val_rows.append({
                    "backbone": b,
                    "dataset": d,
                    "horizon": h,
                    "beta": float(beta),
                    "direct_val_mse": dm,
                    "calibrated_val_mse": mm,
                    "mse_ratio_vs_direct": mm / dm,
                    "gain_vs_direct_pct": gain_pct(dm, mm),
                })

val_curve = pd.DataFrame(val_rows)

assert len(val_curve) == 120 * len(BETA_GRID)
assert val_curve.notna().all().all()

val_curve.to_csv(OUT / "validation_beta_curve_all_conditions.csv", index=False)

print("Rows:", len(val_curve))
display(val_curve.head(20))


In [ ]:

# Cell 7 — select Global-Cal and Dataset-Cal betas using validation only

# One global beta for all 120 conditions.
global_curve = (
    val_curve.groupby("beta", as_index=False)
    .agg(
        macro_mse_ratio=("mse_ratio_vs_direct", "mean"),
        macro_gain_pct=("gain_vs_direct_pct", "mean"),
        median_gain_pct=("gain_vs_direct_pct", "median"),
    )
    .sort_values(["macro_mse_ratio", "beta"], ascending=[True, True])
)

GLOBAL_BETA = float(global_curve.iloc[0]["beta"])

# One beta per dataset, using its 20 validation conditions.
dataset_curve = (
    val_curve.groupby(["dataset", "beta"], as_index=False)
    .agg(
        macro_mse_ratio=("mse_ratio_vs_direct", "mean"),
        macro_gain_pct=("gain_vs_direct_pct", "mean"),
        median_gain_pct=("gain_vs_direct_pct", "median"),
    )
)

best_dataset = (
    dataset_curve
    .sort_values(["dataset", "macro_mse_ratio", "beta"], ascending=[True, True, True])
    .groupby("dataset", as_index=False)
    .first()
)

DATASET_BETA = {
    r.dataset: float(r.beta)
    for r in best_dataset.itertuples(index=False)
}

print("GLOBAL_BETA =", GLOBAL_BETA)
print("DATASET_BETA =", DATASET_BETA)

print("\nGlobal validation curve")
display(global_curve.sort_values("beta"))

print("\nDataset-specific selected betas")
display(best_dataset)

global_curve.to_csv(OUT / "validation_global_beta_curve.csv", index=False)
dataset_curve.to_csv(OUT / "validation_dataset_beta_curve.csv", index=False)
best_dataset.to_csv(OUT / "selected_dataset_betas.csv", index=False)

(OUT / "selected_betas.json").write_text(
    json.dumps(
        {
            "beta_grid": BETA_GRID.tolist(),
            "fixed_beta": FIXED_BETA,
            "global_beta": GLOBAL_BETA,
            "dataset_beta": DATASET_BETA,
            "selection_split": "validation only",
            "selection_objective": "macro mean of condition MSE / Direct MSE",
        },
        indent=2,
    ),
    encoding="utf-8",
)

print("[PASS] beta selection completed without test data")


In [ ]:

# Cell 8 — evaluate Fixed / Global-Cal / Dataset-Cal on TEST

rows = []

for d in DATASETS:
    for h in HORIZONS:
        prr = load_prr(d, h, "test")["retrieval"]

        for b in BACKBONES:
            dt = load_direct(b, d, h, "test")
            direct = dt["direct"]
            truth = dt["truth"]

            dm = mse(direct, truth)
            dmae = mae(direct, truth)

            schemes = {
                "Fixed-0.1": FIXED_BETA,
                "Global-Cal": GLOBAL_BETA,
                "Dataset-Cal": DATASET_BETA[d],
            }

            row = {
                "backbone": b,
                "dataset": d,
                "horizon": h,
                "direct_mse": dm,
                "direct_mae": dmae,
            }

            for name, beta in schemes.items():
                pred = combine(direct, prr, beta)
                mm = mse(pred, truth)
                aa = mae(pred, truth)

                slug = name.lower().replace("-", "_").replace(".", "p")
                row[f"{slug}_beta"] = float(beta)
                row[f"{slug}_mse"] = mm
                row[f"{slug}_mae"] = aa
                row[f"{slug}_gain_vs_direct_pct"] = gain_pct(dm, mm)

            rows.append(row)

test120 = pd.DataFrame(rows)

assert len(test120) == 120
assert test120.notna().all().all()

test120.to_csv(OUT / "test_120_conditions.csv", index=False)
display(test120.head(24))

print("[PASS] test evaluation = 120/120")


In [ ]:

# Cell 9 — exact fixed-beta sanity check against Exp38R3 if available

if EXP38_FIXED.exists():
    ref = pd.read_csv(EXP38_FIXED)
    ref["horizon"] = ref["horizon"].astype(int)

    needed = {
        "backbone", "dataset", "horizon",
        "direct_mse", "learned_fixed01_mse",
    }
    missing = needed - set(ref.columns)

    if missing:
        print("[WARN] Exp38R3 exists but expected columns are missing:", sorted(missing))
    else:
        chk = test120.merge(
            ref[[
                "backbone", "dataset", "horizon",
                "direct_mse", "learned_fixed01_mse",
            ]],
            on=["backbone", "dataset", "horizon"],
            suffixes=("_new", "_ref"),
        )

        assert len(chk) == 120

        max_direct = float(np.max(np.abs(chk.direct_mse_new - chk.direct_mse_ref)))
        max_fixed = float(np.max(np.abs(chk.fixed_0p1_mse - chk.learned_fixed01_mse)))

        print("Max Direct MSE diff   :", max_direct)
        print("Max fixed α=.1 diff   :", max_fixed)

        assert max_direct < 5e-6
        assert max_fixed < 5e-6

        print("[PASS] exact reproduction of validated Exp38R3 fixed-beta results")
else:
    print("[INFO] Exp38R3 fixed result CSV not found; skip exact reference check.")


In [ ]:

# Cell 10 — overall paper-facing summary

scheme_cols = {
    "Fixed-0.1": "fixed_0p1_gain_vs_direct_pct",
    "Global-Cal": "global_cal_gain_vs_direct_pct",
    "Dataset-Cal": "dataset_cal_gain_vs_direct_pct",
}

summary_rows = []

for scheme, col in scheme_cols.items():
    g = test120[col]
    counts = wtl(g)

    summary_rows.append({
        "scheme": scheme,
        "conditions": len(g),
        **counts,
        "mean_gain_vs_direct_pct": float(g.mean()),
        "median_gain_vs_direct_pct": float(g.median()),
        "min_gain_pct": float(g.min()),
        "max_gain_pct": float(g.max()),
    })

overall = pd.DataFrame(summary_rows)

display(overall)
overall.to_csv(OUT / "overall_summary.csv", index=False)

print("\nSelected α")
print("Fixed       :", FIXED_BETA)
print("Global-Cal  :", GLOBAL_BETA)
print("Dataset-Cal :", DATASET_BETA)


In [ ]:

# Cell 11 — dataset-level test summary

rows = []

for d, gdf in test120.groupby("dataset", sort=False):
    for scheme, col in scheme_cols.items():
        gains = gdf[col]
        counts = wtl(gains)

        rows.append({
            "dataset": d,
            "scheme": scheme,
            "beta": (
                FIXED_BETA if scheme == "Fixed-0.1"
                else GLOBAL_BETA if scheme == "Global-Cal"
                else DATASET_BETA[d]
            ),
            "conditions": len(gains),
            **counts,
            "mean_gain_vs_direct_pct": float(gains.mean()),
            "median_gain_vs_direct_pct": float(gains.median()),
        })

dataset_summary = pd.DataFrame(rows)

display(dataset_summary)
dataset_summary.to_csv(OUT / "dataset_summary.csv", index=False)


In [ ]:

# Cell 12 — backbone / horizon summaries for the candidate main result (Dataset-Cal)

def grouped_summary(df, group_col, gain_col):
    rows = []
    for key, gdf in df.groupby(group_col, sort=False):
        gains = gdf[gain_col]
        counts = wtl(gains)
        rows.append({
            group_col: key,
            "conditions": len(gains),
            **counts,
            "mean_gain_vs_direct_pct": float(gains.mean()),
            "median_gain_vs_direct_pct": float(gains.median()),
        })
    return pd.DataFrame(rows)

backbone_summary = grouped_summary(
    test120, "backbone", "dataset_cal_gain_vs_direct_pct"
)
horizon_summary = grouped_summary(
    test120, "horizon", "dataset_cal_gain_vs_direct_pct"
)

print("DATASET-CAL — BACKBONE SUMMARY")
display(backbone_summary)

print("DATASET-CAL — HORIZON SUMMARY")
display(horizon_summary)

backbone_summary.to_csv(OUT / "dataset_cal_backbone_summary.csv", index=False)
horizon_summary.to_csv(OUT / "dataset_cal_horizon_summary.csv", index=False)


In [ ]:

# Cell 13 — validation sensitivity table around the selected betas

# Useful for the appendix: shows calibration is not a test-set artifact.
global_sensitivity = global_curve.sort_values("beta").copy()

dataset_sensitivity = (
    dataset_curve
    .pivot(index="dataset", columns="beta", values="macro_mse_ratio")
    .reset_index()
)

print("GLOBAL validation normalized MSE ratio (lower is better)")
display(global_sensitivity)

print("DATASET validation normalized MSE ratio")
display(dataset_sensitivity)

dataset_sensitivity.to_csv(
    OUT / "validation_dataset_sensitivity_matrix.csv",
    index=False,
)


In [ ]:

# Cell 14 — final decision readout (fixed pandas Series column access)

def row_for(name):
    x = overall[overall["scheme"] == name]
    assert len(x) == 1
    return x.iloc[0]

fixed = row_for("Fixed-0.1")
glob = row_for("Global-Cal")
dcal = row_for("Dataset-Cal")

def pack_summary(row):
    return {
        "W": int(row["W"]),
        "T": int(row["T"]),
        "L": int(row["L"]),
        "mean_gain_pct": float(row["mean_gain_vs_direct_pct"]),
        "median_gain_pct": float(row["median_gain_vs_direct_pct"]),
    }

decision = {
    "experiment": "Exp40",
    "training_performed": False,
    "forecaster_inference_performed": False,
    "test_used_for_beta_selection": False,
    "beta_grid": BETA_GRID.tolist(),
    "fixed_beta": float(FIXED_BETA),
    "global_beta": float(GLOBAL_BETA),
    "dataset_beta": {k: float(v) for k, v in DATASET_BETA.items()},

    "fixed": pack_summary(fixed),
    "global_cal": pack_summary(glob),
    "dataset_cal": pack_summary(dcal),
}

(OUT / "exp40_summary.json").write_text(
    json.dumps(decision, indent=2),
    encoding="utf-8",
)

print(json.dumps(decision, indent=2))

print("\n" + "="*100)
print("PAPER DECISION GUIDE")
print("="*100)
print(
    "If Dataset-Cal materially improves over Fixed-0.1 "
    "(e.g., clearly more wins and larger mean gain), "
    "it is eligible to become the main downstream result because "
    "beta was selected strictly on validation."
)
print(
    "Keep Fixed-0.1 in the paper as a controlled ablation/control "
    "to demonstrate that the calibrated result is not produced by test tuning."
)
print(
    "If improvement is only marginal, retain Fixed-0.1 as the main result "
    "and report calibration in the appendix."
)
print("\nSaved under:", OUT)
